# Raw json EDA

In [ ]:
from pathlib import Path
import json
import pandas as pd

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)

## 1) 데이터 분포 확인

In [7]:
def resolve_raw_dir() -> Path:
    candidates = [
        Path.cwd() / 'data' / 'raw',
        Path.cwd().parent / 'data' / 'raw',
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError('Could not find data/raw directory.')


RAW_DIR = resolve_raw_dir()

DATA_FILES = [
    RAW_DIR / 'TL_피부과_통합.json',
    RAW_DIR / 'VL_피부과_통합.json',
]

def load_df(path: Path) -> pd.DataFrame:
    # Source JSON has BOM, so use utf-8-sig.
    records = json.loads(path.read_text(encoding='utf-8-sig'))
    return pd.DataFrame(records)


def text_len_series(series: pd.Series) -> pd.Series:
    return series.fillna('').astype(str).str.len()


def print_len_distribution(name: str, series: pd.Series) -> None:
    bins = [-1, 30, 60, 90, 120, 200, 300, 500, 1000, 10**9]
    labels = ['0-30', '31-60', '61-90', '91-120', '121-200', '201-300', '301-500', '501-1000', '1001+']
    binned = pd.cut(series, bins=bins, labels=labels)

    print(f'\n[{name} length summary]')
    print(series.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(2))

    print(f'\n[{name} length bin distribution]')
    print(binned.value_counts(sort=False))


for path in DATA_FILES:
    df = load_df(path)

    print('\n' + '=' * 80)
    print(f'dataset: {path.name}')
    print('=' * 80)

    print('\n[qa_id counts]')
    print(f'- total rows: {len(df):,}')
    print(f'- unique qa_id: {df["qa_id"].nunique():,}')

    print('\n[q_type distribution]')
    q_type_counts = df['q_type'].value_counts(dropna=False)
    q_type_ratio = (q_type_counts / len(df) * 100).round(2).astype(str) + '%'
    q_type_dist = pd.DataFrame({'count': q_type_counts, 'ratio': q_type_ratio})
    print(q_type_dist)

    q_len = text_len_series(df['question'])
    a_len = text_len_series(df['answer'])

    print_len_distribution('question', q_len)
    print_len_distribution('answer', a_len)


dataset: TL_피부과_통합.json

[qa_id counts]
- total rows: 612
- unique qa_id: 612

[q_type distribution]
        count   ratio
q_type               
1         419  68.46%
2          97  15.85%
3          96  15.69%

[question length summary]
count    612.00
mean     201.59
std       86.99
min       23.00
25%      136.75
50%      186.00
75%      260.00
90%      320.80
95%      352.00
99%      420.12
max      567.00
Name: question, dtype: float64

[question length bin distribution]
question
0-30          1
31-60        13
61-90        30
91-120       67
121-200     230
201-300     182
301-500      86
501-1000      3
1001+         0
Name: count, dtype: int64

[answer length summary]
count    612.00
mean      55.81
std       91.66
min        2.00
25%       10.00
50%       16.50
75%       37.00
90%      240.90
95%      287.45
99%      339.90
max      431.00
Name: answer, dtype: float64

[answer length bin distribution]
answer
0-30        429
31-60        81
61-90         9
91-120        2
121-

## 2) 전문 의학용어 추출

In [23]:
import re
from collections import Counter

# Match: Korean term + (English term), allowing missing closing bracket in source.
KO_EN_PATTERN = re.compile(
    r"([\uac00-\ud7a3][\uac00-\ud7a3A-Za-z0-9\u00b7\-\s]{1,60}?)\s*[\(\[]\s*([A-Za-z][A-Za-z0-9\-\./\s]{1,50})\s*[\)\]]?"
)
OPTION_MARK_PATTERN = re.compile(r"[0-9]+\)\s*")

KOR_TERM_SUFFIXES = (
    "피부염", "염", "증", "병", "암", "종", "증후군", "루푸스",
    "건선", "백선", "천포창", "관절염", "육종", "림프종"
)
VERB_LIKE_ENDINGS = (
    "한다", "하였다", "합니다", "된다", "줄인다",
    "사용한다", "사용하는 것", "차이가 없다", "없다", "보인다"
)


def normalize_term(term: str) -> str:
    term = " ".join(str(term).replace("\u3000", " " ).split())
    term = term.strip(" ,.;:!?)([]")
    return term


def strip_unbalanced_brackets(term: str) -> str:
    term = term.strip()
    while term and term[-1] in "([":
        term = term[:-1].strip()
    return term


def looks_like_sentence(term: str) -> bool:
    if len(term) >= 24 and term.count(" ") >= 3:
        return True
    if any(term.endswith(e) for e in VERB_LIKE_ENDINGS):
        return True
    if any(phrase in term for phrase in ["치료 반응", "매일 사용하는 것", "복용", "투여", "개선하여", "예방한다"]):
        return True
    return False


def is_term_candidate(term: str) -> bool:
    if not term:
        return False

    term = strip_unbalanced_brackets(term)
    if len(term) < 2 or len(term) > 60:
        return False
    if not re.search(r"[\uac00-\ud7a3A-Za-z]", term):
        return False
    if any(ch in term for ch in "?!"):
        return False
    if looks_like_sentence(term):
        return False

    # Overly long Korean phrases are usually explanatory text, not terms.
    if re.search(r"[\uac00-\ud7a3]", term) and term.count(" ") >= 4 and not any(term.endswith(s) for s in KOR_TERM_SUFFIXES):
        return False

    return True


def extract_option_terms(text: str):
    terms = []
    for line in text.splitlines():
        if ")" not in line:
            continue

        parts = OPTION_MARK_PATTERN.split(line)
        if len(parts) <= 1:
            continue

        for chunk in parts[1:]:
            candidate = strip_unbalanced_brackets(normalize_term(chunk))
            if is_term_candidate(candidate):
                terms.append(candidate)
    return terms


def extract_terms_and_mapping_from_record(question: str, answer: str):
    terms = []
    mappings = {}
    text = f"{question}\n{answer}"

    # 1) MCQ-style option phrases
    terms.extend(extract_option_terms(text))

    # 2) Short free-text answer
    if ")" not in answer:
        ans = strip_unbalanced_brackets(normalize_term(answer))
        if is_term_candidate(ans):
            terms.append(ans)

    # 3) Korean-English pair mapping
    for ko_raw, en_raw in KO_EN_PATTERN.findall(text):
        ko = strip_unbalanced_brackets(normalize_term(ko_raw.split(":")[-1]))
        en = strip_unbalanced_brackets(normalize_term(en_raw))

        if not is_term_candidate(ko):
            continue
        if not is_term_candidate(en):
            continue

        terms.append(ko)
        terms.append(en)
        mappings[ko] = en

    return terms, mappings


def postprocess_counter(counter: Counter) -> dict:
    filtered = Counter()
    for term, freq in counter.items():
        cleaned = strip_unbalanced_brackets(normalize_term(term))
        if is_term_candidate(cleaned):
            filtered[cleaned] += freq

    sorted_items = sorted(filtered.items(), key=lambda x: (-x[1], x[0]))
    return dict(sorted_items)


def postprocess_mapping(mapping: dict) -> dict:
    cleaned_map = {}
    for ko, en in mapping.items():
        ko_c = strip_unbalanced_brackets(normalize_term(ko))
        en_c = strip_unbalanced_brackets(normalize_term(en))

        if not is_term_candidate(ko_c):
            continue
        if not is_term_candidate(en_c):
            continue
        if looks_like_sentence(ko_c) or looks_like_sentence(en_c):
            continue

        cleaned_map[ko_c] = en_c
    return cleaned_map


def build_term_dict_and_mapping(json_path: Path):
    records = json.loads(json_path.read_text(encoding="utf-8-sig"))
    counter = Counter()
    mapping = {}

    for row in records:
        question = row.get("question", "")
        answer = row.get("answer", "")
        terms, mappings = extract_terms_and_mapping_from_record(question, answer)
        counter.update(terms)
        mapping.update(mappings)

    term_dict = postprocess_counter(counter)
    map_dict = postprocess_mapping(mapping)
    return term_dict, map_dict

def save_dict_json(terms: dict, file_name: str):
    with open(file_name, "w", encoding="utf-8") as f:
        json.dump(terms, f, ensure_ascii=False, indent=4)

In [21]:
TL_PATH = RAW_DIR / 'TL_피부과_통합.json'
VL_PATH = RAW_DIR / 'VL_피부과_통합.json'

tl_dict, tl_mapping = build_term_dict_and_mapping(TL_PATH)
vl_dict, vl_mapping = build_term_dict_and_mapping(VL_PATH)

# 3) Combined mapping from TL and VL
ko_en_mapping_dict = {}
ko_en_mapping_dict.update(tl_mapping)
ko_en_mapping_dict.update(vl_mapping)

print(f"TL terms: {len(tl_dict):,}")
print(f"VL terms: {len(vl_dict):,}")
print(f"KO-EN mappings: {len(ko_en_mapping_dict):,}")

print("\nTop 20 TL terms:")
print(dict(list(tl_dict.items())[:20]))

print("\nTop 20 VL terms:")
print(dict(list(vl_dict.items())[:20]))

print("\nTop 20 ko_en mappings:")
print(dict(list(ko_en_mapping_dict.items())[:20]))

TL terms: 1,602
VL terms: 203
KO-EN mappings: 295

Top 20 TL terms:
{'건선': 26, '아토피 피부염': 17, '접촉성 피부염': 16, '백반증': 15, '피부 생검': 15, '국소 스테로이드': 12, '지루성 피부염': 11, '방사선 치료': 9, 'BCC': 8, '기저세포암': 8, '레이저 치료': 8, '접촉 피부염': 8, '항히스타민제': 8, 'Staphylococcus aureus': 7, '아토피피부염': 7, '전기소작술': 7, '테르비나핀': 7, '편평세포암': 7, 'Pseudomonas aeruginosa': 6, 'psoriasis': 6}

Top 20 VL terms:
{'기저세포암': 3, 'Amidoamine (AA': 2, 'IL-5': 2, 'Kaposie-Stemmer 징후': 2, 'PUVA 광 치료': 2, 'Psoriatic arthritis': 2, 'SPF': 2, 'Scabies': 2, 'dermographism': 2, 'neurocutaneous melanosis': 2, '각질형성세포': 2, '건선성 관절염': 2, '국소 스테로이드 크림': 2, '그리세오풀빈 6~8주': 2, '높은 선 프로텍션 팩터': 2, '두피(특히 두개골 정점': 2, '두피에서 살아있는 머릿니 관찰': 2, '리팜피신과 클라리스로마이신 병용 요법': 2, '비소 중독': 2, '세균': 2}

Top 20 ko_en mappings:
{'전신홍반루푸스': 'Systemic Lupus Erythematosus', '주사': 'Rosacea', '손바닥에서는 큰어귀': 'Thenar', '새끼어귀': 'Hypothenar', '혈전성 혈소판감소성 자반증': 'Thrombotic thrombocytopenic purpura', '베체트병': 'Beh', '전신경화증': 'Systemic sclerosis', '라이터 증후군': 'Reiter', '단순포진': 

## 3) 딕셔너리 json file 저장

In [25]:
OUTPUT_DIR = Path.cwd().parent / 'data' / 'processed'

OUTPUT_TASKS = [
    (tl_dict, OUTPUT_DIR / 'TL_terms.json'),
    (vl_dict, OUTPUT_DIR / 'VL_terms.json'),
    (ko_en_mapping_dict, OUTPUT_DIR / 'ko_en_mappings.json'),
]

for data, path in OUTPUT_TASKS:
    save_dict_json(data, path)